# Simple Power Analysis for Password Bypass

** This is based on the notebook from ChipWhisperer's Part 2, Topic 1, Lab B: Power Analysis for Password Bypass (HARDWARE) with some minor changes. 
** The main purpose of this lab is to demonstrate distinguishable power patterns between a correct char and an incorrect char in the first char of a password. 

---
NOTE: This lab references some (commercial) training material on [ChipWhisperer.io](https://www.ChipWhisperer.io). You can freely execute and use the lab per the open-source license (including using it in your own courses if you distribute similarly), but you must maintain notice about this source location. Consider joining our training course to enjoy the full experience.

---

**SUMMARY:** *This tutorial will introduce you to breaking devices by determining when a device is performing certain operations. Our target device will be performing a simple password check, and we will demonstrate how to perform a basic power analysis.*

**LEARNING OUTCOMES:**

* How power can be used to determine timing information.
* Plotting multiple iterations while varying input data to find interesting locations.
* Using difference of waveforms to find interesting locations.
* Performing power captures with ChipWhisperer hardware (hardware only)

## Prerequisites

Hold up! Before you continue, check you've done the following tutorials:

* ☑ Jupyter Notebook Intro (you should be OK with plotting & running blocks).
* ☑ SCA101 Intro (you should have an idea of how to get hardware-specific versions running).

## (STM32F) Setup for STM32F3

First you'll need to select which hardware setup you have. You'll need to select both a `SCOPETYPE` and a `PLATFORM`. `SCOPETYPE` can either be `'OPENADC'` for the CWLite/CW1200 or `'CWNANO'` for the CWNano. `PLATFORM` is the target device, with `'CWLITEARM'`/`'CW308_STM32F3'` being the best supported option, followed by `'CWLITEXMEGA'`/`'CW308_XMEGA'`, then by `'CWNANO'`. As of CW 5.4, you can select the SimpleSerial version
used. For example:

```python
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'
```

In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CWLITEARM'
SS_VER = 'SS_VER_2_1'

## Detect, Compile, and Upload to ChipWhisperer

This code will connect the scope and do some basic setup. We're now just going to use a special setup script to do this. This script contains the commands we ran seperately before. Update username 'boyang' to your username or update the path accordingly for your system. 

In [2]:
%run "/home/boyang/chipwhisperer/jupyter/Setup_Scripts/Setup_Generic.ipynb"

INFO: Found ChipWhisperer😍
scope.gain.mode                          changed from low                       to high                     
scope.gain.gain                          changed from 0                         to 30                       
scope.gain.db                            changed from 5.5                       to 24.8359375               
scope.adc.basic_mode                     changed from low                       to rising_edge              
scope.adc.samples                        changed from 24400                     to 5000                     
scope.adc.trig_count                     changed from 16716573                  to 38847452                 
scope.clock.adc_src                      changed from clkgen_x1                 to clkgen_x4                
scope.clock.adc_freq                     changed from 0                         to 29538459                 
scope.clock.adc_rate                     changed from 0.0                       to 29538459.0        

The following code will build the firmware for the target. Update username 'boyang' to your username or update the path accordingly for your system.

In [3]:
%%bash -s "$PLATFORM" "$SS_VER"
cd /home/boyang/chipwhisperer/firmware/mcu/basic-passwdcheck
make PLATFORM=$1 CRYPTO_TARGET=NONE SS_VER=$2 -j

SS_VER set to SS_VER_2_1
SS_VER set to SS_VER_2_1
arm-none-eabi-gcc (15:10.3-2021.07-4) 10.3.1 20210621 (release)
Copyright (C) 2020 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

mkdir -p objdir-CWLITEARM 
.
Welcome to another exciting ChipWhisperer target build!!
.
.
.
Compiling:
Compiling:
Compiling:
-en     basic-passwdcheck.c ...
-en     .././simpleserial/simpleserial.c ...
-en     .././hal/hal.c ...
.
Compiling:
.
.
-en     .././hal//stm32f3/stm32f3_hal.c ...
Compiling:
Compiling:
-en     .././hal//stm32f3/stm32f3_hal_lowlevel.c ...
-en     .././hal//stm32f3/stm32f3_sysmem.c ...
.
Assembling: .././hal//stm32f3/stm32f3_startup.S
arm-none-eabi-gcc -c -mcpu=cortex-m4 -I. -x assembler-with-cpp -mthumb -mfloat-abi=soft -fmessage-length=0 -ffunction-sections -DF_CPU=7372800 -Wa,-gstabs,-adhlns=objdir-CWLITEARM/stm32f3_startup.lst -I.././simpleserial/ -

Finally, all that's left is to program the device, which can be done with the following line: Update username 'boyang' to your username or update the path accordingly for your system.

In [4]:
cw.program_target(scope, prog, "/home/boyang/chipwhisperer/firmware/mcu/basic-passwdcheck/basic-passwdcheck-{}.hex".format(PLATFORM))

Detected known STMF32: STM32F302xB(C)/303xB(C)
Extended erase (0x44), this can take ten seconds or more
Attempting to program 4811 bytes at 0x8000000
STM32F Programming flash...
STM32F Reading flash...
Verified flash OK, 4811 bytes


## A Look at the Target Program

For this lab, our goal is to get the following code to run and obtain a trace for each given password we provide. The source code can be found in 

/chipwhisperer/firmware/mcu/basic-passwdcheck/basic-passwdcheck.c

```python

int main(void) {
    platform_init();
    init_uart();
    trigger_setup();

    char passwd[32];
    char correct_passwd[] = "h0px3";

    while(1) {
        my_puts("*****Safe-o-matic 3000 Booting...\n");
        my_puts("Decrypting database..[DONE]\n");
        delay_2_ms();
        my_puts("\n\n");
        //Give them one last warning
        my_puts("WARNING: UNAUTHORIZED ACCESS WILL BE PUNISHED\n");

        trigger_low();

        //Get password
        my_puts("Please enter password to continue: ");
        my_read(passwd, 32);
        uint8_t passbad = 0;

        trigger_high();

        for(uint8_t i = 0; i < sizeof(correct_passwd); i++){
            if (correct_passwd[i] != passwd[i]){
                passbad = 1;
                break;
            }
        }

        if (passbad){
            // Stop them fancy timing attacks
            int wait = 1;
            for(volatile int i = 0; i < wait; i++){
                ;
            }
            delay_2_ms();
            delay_2_ms();
            my_puts("PASSWORD FAIL\n");
            led_error(1);
        } else {
            my_puts("Access granted, Welcome!\n");
            led_ok(1);
        }
        
        //All done;
        while(1);
  }

  return 1;
}
```

As we can see from the above program, the real password ('h0px3') is hard-coded in the program for the ease of demonstration. If the given password is correct, the varialbe passbad remains to be 0. Otherwise, passbad will be updated to 1.  

From the side-channel analysis perspective, since the verification is based on each char. If one char is correct, it will move on verifying the next char. Otherwise, if the char is not correct, the program will break without verifying the next char. In other words, the pattern of power traces from all the incorrect first chars will be the same or similar but the pattern of the power trace from the correct first char will be distinguishable from others. 

## Setup Trace Collection Function

To make interacting with the hardware easier, let's define a function to attempt a password and return a power trace:

In [7]:
def cap_pass_trace(pass_guess):
    reset_target(scope)
    num_char = target.in_waiting()
    while num_char > 0:
        response = target.read(num_char, 10)
        time.sleep(0.01)
        num_char = target.in_waiting()
        print(num_char)
        print(response)

    scope.arm()
    target.write(pass_guess)
    ret = scope.capture()
    if ret:
        print('Timeout happened during acquisition')

    trace = scope.get_last_trace()
    return trace

We also don't need all of the default 5000 samples in the trace. 150 is a good starting point for the executions associated with the verification on the first character of this target password program:

In [5]:
scope.adc.samples = 150

## Sending a password with a single char 'h'

In [9]:
trace_test1 = cap_pass_trace("h\n")

#Basic sanity check
assert(len(trace_test1) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test1).opts(color='blue')

0
PASSWORD FAIL
 *****Safe-o-matic 3000 Booting...
Decrypting database..[DONE]


Please enter password to continue: 
✔️ OK to continue!


:Curve   [x]   (y)

## Sending a password with a single char 'a'

In [10]:
trace_test2 = cap_pass_trace("a\n")

#Basic sanity check
assert(len(trace_test2) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test2).opts(color='green')

0
PASSWORD FAIL
 *****Safe-o-matic 3000 Booting...
Decrypting database..[DONE]


Please enter password to continue: 
✔️ OK to continue!


:Curve   [x]   (y)

## Sending a password with a single char 'b'

In [11]:
trace_test3 = cap_pass_trace("b\n")

#Basic sanity check
assert(len(trace_test3) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test3).opts(color='red')

0
PASSWORD FAIL
 *****Safe-o-matic 3000 Booting...
Decrypting database..[DONE]


Please enter password to continue: 
✔️ OK to continue!


:Curve   [x]   (y)

## Sending a password with a single char 'c'

In [12]:
trace_test4 = cap_pass_trace("c\n")

#Basic sanity check
assert(len(trace_test4) == 150)
print("✔️ OK to continue!")

cw.plot(trace_test4).opts(color='orange')

0
PASSWORD FAIL
 *****Safe-o-matic 3000 Booting...
Decrypting database..[DONE]


Please enter password to continue: 
✔️ OK to continue!


:Curve   [x]   (y)

## Plot all traces in a single plot to distinguish the one from incorrect guess

In [13]:
fig = cw.plot(trace_test1).opts(color='blue') 
fig *= cw.plot(trace_test2).opts(color='green')
fig *= cw.plot(trace_test3).opts(color='red')
fig *= cw.plot(trace_test4).opts(color='orange')
fig

:Overlay
   .Curve.I   :Curve   [x]   (y)
   .Curve.II  :Curve   [x]   (y)
   .Curve.III :Curve   [x]   (y)
   .Curve.IV  :Curve   [x]   (y)